In [43]:
import pandas as pd
import numpy as np
import re

In [19]:
df = pd.read_csv(r"C:\Users\avanindra Bose\OneDrive\Desktop\Real State Project\Cleaned Datasets\gurgaon_properties_cleaned_v1.csv")

In [20]:
df.head(1)

,property_type,society,sector,price,price_per_square_feet,area,areaWithType,bedRoom,bathroom,balcony,additionalRoom,floorNum,facing,agePossession,nearbyLocations,furnishDetails,features
0,flat,rof aalayas,sector 102,0.45,6000.0,750.0,Built Up area: 750 (69.68 sq.m.),1,1,2,not available,6.0,NaN,undefined,"['HUDA Metro Station', 'Idea Cosmic Plaza', 'Gurugram Road', 'Dwarka Expy', 'GEMS International School', 'The NorthCap University', 'Aryan Hospital', 'Indira Gandhi Intl Airport']","['1 Light', 'No AC', 'No Bed', 'No Chimney', 'No Curtains', 'No Dining Table', 'No Exhaust Fan', 'No Fan', 'No Geyser', 'No Modular Kitchen', 'No Microwave', 'No Fridge', 'No Sofa', 'No Stove', 'No TV', 'No Wardrobe', 'No Washing Machine', 'No Water Purifier']",NaN


**There are 5 Columns using which we can extract some important information :**

1.areaWithType

2.additionalRoom

3.agepossession

4.furnish Details

5.features

Handling Area with Type Column

In [42]:
df[['area','areaWithType']].sample(5)

,area,areaWithType
3649,1800.0,Plot area 200(167.23 sq.m.)
2715,2965.0,Super Built up area 2965(275.46 sq.m.)
2051,1245.0,Super Built up area 1245(115.66 sq.m.)Built Up area: 1154 sq.ft. (107.21 sq.m.)Carpet area: 945 sq.ft. (87.79 sq.m.)
2706,4739.0,Super Built up area 4739(440.27 sq.m.)Built Up area: 3655.35 sq.ft. (339.59 sq.m.)
428,1929.0,Super Built up area 1929(179.21 sq.m.)Built Up area: 1550 sq.ft. (144 sq.m.)Carpet area: 1330 sq.ft. (123.56 sq.m.)


The Problem here is , We have 3 types of area Carpet Area , BuiltUp Area and Super Built Up Area for flats and for houses/Indepenedent houses we have plot area. 

And the Units for all the Areas are different , but we should follow sqft. (For all the area calculations)

As a Part of our solution our first step should be fetching the three different areas -> BuiltUpArea , Carpet Area , Super Built Up Area

In [44]:
def get_super_built_up_area(text):
    match = re.search(r'Super Built up area (\d+\.?\d*)' , text)
    if match :
        return float(match.group(1))
    return None

In [60]:
def get_area(text, area_type):
    match = re.search(area_type + r'\s*:\s*(\d+\.?\d*)', text)
    if match:
        return float(match.group(1))
    return None

Conversion into Sqft values

In [50]:
def convert_to_sqft(text,area_value):
    if area_value is None :
        return None
    match = re.search(r'{} \((\d+\.?\d*) sq.m.\)'.format(area_value), text)
    if match:
        sq_m_value = float(match.group(1))
        return sq_m_value * 10.7639  # conversion factor from sq.m. to sqft
    return area_value

In [55]:
df['super_built_up_area'] = df['areaWithType'].apply(get_super_built_up_area)
df['super_built_up_area'] = df.apply(lambda row: convert_to_sqft(row['areaWithType'], row['super_built_up_area']), axis=1)

In [61]:
df['built_up_area'] = df['areaWithType'].apply(lambda x: get_area(x, 'Built Up area'))
df['built_up_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['built_up_area']), axis=1)

In [62]:
df['carpet_area'] = df['areaWithType'].apply(lambda x: get_area(x, 'Carpet area'))
df['carpet_area'] = df.apply(lambda x: convert_to_sqft(x['areaWithType'], x['carpet_area']), axis=1)

In [58]:
df.drop(columns=['super_built_up_area_sqft'], inplace=True)

In [73]:
df[['price','property_type','area','areaWithType','super_built_up_area','built_up_area','carpet_area']].sample(5)

,price,property_type,area,areaWithType,super_built_up_area,built_up_area,carpet_area
2113,5.88,house,2160.0,Plot area 240(200.67 sq.m.),NaN,NaN,NaN
37,4.30,house,175.0,Plot area 163(15.14 sq.m.)Built Up area: 145 sq.ft. (13.47 sq.m.),NaN,145.0,NaN
2637,1.80,flat,1889.0,Super Built up area 1889(175.49 sq.m.),1889.0,NaN,NaN
1191,0.41,flat,594.0,Carpet area: 601 (55.83 sq.m.),NaN,NaN,601.0
1713,1.00,flat,1554.0,Super Built up area 1554(144.37 sq.m.),1554.0,NaN,NaN


In [74]:
df[~((df['super_built_up_area'].isnull()) | (df['built_up_area'].isnull()) | (df['carpet_area'].isnull()))][['price','property_type','area','areaWithType','super_built_up_area','built_up_area','carpet_area']].shape

(534, 7)

Where all the 3 features super_built_up_area	built_up_area	carpet_area are NAN it is a house data meaning it will have a plot area